In [16]:
import pandas as pd
import numpy as np
from collections import Counter

##### Assumptions
- 1 year of a tender is allocated 1 year of antigen demand (through 1 or many vaccines); scales appropriately
- Intenvory is based on demand for a given year.
- Ratio of inventory to supply is calculated based on end of year totals (after calcs), but only in the programatic sense (check after math, add inventory at end of year for next year)

##### Load and setup demand

In [17]:
# Load the CSV file
demand_path = 'data/real/antigen_demand_80_20_2_scenarios.csv'
data = pd.read_csv(demand_path)
# Create the two dataframes based on the 'prob' column
demand_80 = data[data['prob'] == 0.8]
demand_20 = data[data['prob'] == 0.2]

demand_80 = demand_80.drop(columns=['prob', 'demand_SID'])
demand_20 = demand_20.drop(columns=['prob', 'demand_SID'])
# Expanding the 'demands' column into 10 separate columns
demand_80_expanded = demand_80['demands'].apply(lambda x: pd.Series(eval(x)))
demand_20_expanded = demand_20['demands'].apply(lambda x: pd.Series(eval(x)))

# Renaming the columns to 1-10
demand_80_expanded.columns = range(1, 11)
demand_20_expanded.columns = range(1, 11)

# Concatenating the expanded demands columns back to the original antigen column
demand_80_final = pd.concat([demand_80['antigen'], demand_80_expanded], axis=1)
#added 11th year to capture any left overdemand at the end of year 10.
demand_80_final[11] = 0.1
demand_20_final = pd.concat([demand_20['antigen'], demand_20_expanded], axis=1)
#added 11th year to capture any left overdemand at the end of year 10.
demand_20_final[11] = 0.1

# demand_80_final.head(), demand_20_final.head()


##### Load and setup Starting Points

In [18]:
file_path = 'data/real/Starting_point.xlsx'

# Load the sheets 'F_start', 'I_start', 'S_start' into their own DataFrames
f_start = pd.read_excel(file_path, sheet_name='F_start')
i_start = pd.read_excel(file_path, sheet_name='I_start')
missed_doses = pd.read_excel(file_path, sheet_name='S_start')


#need to go back throug and re-assign F-start and I-start to new nemes so we dont create extra copies

##### Load pricing data

In [33]:
# Load the Excel file, skipping the first two sheets
money_path = 'data/Vaccine_price_data.xlsx'
sheet_names = pd.ExcelFile(money_path).sheet_names

# Load the remaining sheets into a dictionary of DataFrames
# data = {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in sheet_names[2:5]}
price_data = {sheet_names[i]: pd.read_excel(money_path, sheet_name=i).rename(columns=lambda x: "Producer" if x == pd.read_excel(money_path, sheet_name=i).columns[0] else x) for i in range(2, 5)}

# Display the names of the loaded sheets and the first few rows of the first sheet
sheet_names_loaded = list(price_data.keys())
# first_sheet_preview = data[sheet_names_loaded[0]].head()


In [34]:
price_data

{'M Pricing':           Producer      1       2     3        4         5         6  \
 0           PT_Bio  0.250  0.2600  0.27  0.26500  0.267500  0.266250   
 1  Serum_Institute  0.392  0.4305  0.44  0.43525  0.437625  0.436438   
 
           7         8         9        10  
 0  0.266875  0.266562  0.266719  0.266641  
 1  0.437031  0.436734  0.436883  0.436809  ,
 'MR Pricing':           Producer       1       2     3        4         5         6  \
 0  Serum_Institute  0.8115  0.8925  0.89  0.89125  0.890625  0.890938   
 1     Biological_E  0.6995  0.8190  0.87  0.94450  0.907250  0.925875   
 
           7         8         9        10  
 0  0.890781  0.890859  0.890820  0.890840  
 1  0.916562  0.921219  0.918891  0.920055  ,
 'MMR Pricing':           Producer      1         2         3         4         5         6  \
 0  Serum_Institute  1.944  2.136333  2.348333  2.242333  2.295333  2.268833   
 1              GSK  4.470  4.470000  4.470000  4.470000  4.470000  4.470000   
 

##### Load capacity data

In [31]:
# Load the Excel file, only reading the first sheet
capacity_path = 'data/production_capacity_scenarios.xlsx'
sheet_names = pd.ExcelFile(capacity_path).sheet_names

capacity_data = pd.read_excel(capacity_path, sheet_name='base_capacity')
capacity_data

,Manufacturer,1,2,3,4,5,6,7,8,9,10
0,AJ_Vaccines,7711003,7711003,7711003,7711003,7711003,7711003,7711003,7711003,7711003,7711003
1,BB_NCIPD,39223956,39223956,39223956,39223956,39223956,39223956,39223956,39223956,39223956,39223956
2,Bharat_Biotech,61029105,61029105,61029105,61029105,61029105,61029105,61029105,61029105,61029105,61029105
3,Bilthoven,12048153,12048153,12048153,12048153,12048153,12048153,12048153,12048153,12048153,12048153
4,Biological_E,164885690,164885690,164885690,164885690,164885690,164885690,164885690,164885690,164885690,164885690
5,China_National,12812242,12812242,12812242,12812242,12812242,12812242,12812242,12812242,12812242,12812242
6,GSK,226762686,226762686,226762686,226762686,226762686,226762686,226762686,226762686,226762686,226762686
7,Haffkine_Bio,80972855,80972855,80972855,80972855,80972855,80972855,80972855,80972855,80972855,80972855
8,LG_Chem,43702188,43702188,43702188,43702188,43702188,43702188,43702188,43702188,43702188,43702188
9,Merck_Sharp,56991052,56991052,56991052,56991052,56991052,56991052,56991052,56991052,56991052,56991052


##### Initialize stuff

In [21]:
#tender length
delta = 3
vaccine_consumption_percent = 1
years = 10

# Creating an empty DataFrame with the specified structure for calculating ratios
antigens = f_start['Antigen']
columns = ['Antigen',1]

ratio_DF = pd.DataFrame(columns=columns)
ratio_DF['Antigen'] = antigens
ratio_DF[1] = np.zeros(len(antigens))

# create DF to store tender schedule
tender_schedules = f_start.copy()

########################################################
#create DF to store current inventory
inventory_DF = i_start.copy()
##########################################################
#create DF to store unvaccinated children


In [22]:
def calculate_coverage_and_ratios(inventory_DF, demand_DF, V_a, year):
    # Initialize the total coverage dictionary
    total_coverage = {antigen: 0 for antigen in V_a.keys()}

    # Iterate across each row in the inventory DataFrame
    for index, row in inventory_DF.iterrows():
        vaccine = row['Vaccine']
        amount = row['Amount']
        
        # For each antigen covered by the vaccine, add the amount to the coverage
        for antigen in V_a.keys():
            if vaccine in V_a[antigen]:
                total_coverage[antigen] += amount

    # Convert the total coverage dictionary to a DataFrame
    total_coverage_df = pd.DataFrame(list(total_coverage.items()), columns=['antigen', 'Total_Coverage'])
    # print(total_coverage_df)

    # Calculate ratio of supply and demand
    calculate_ratios_DF = pd.merge(demand_DF.iloc[:, [0, year]], total_coverage_df, left_on='antigen', right_on='antigen')
    # print(calculate_ratios_DF)
    calculate_ratios_DF['Ratio'] = calculate_ratios_DF.apply(lambda row: 0 if row['Total_Coverage'] == 0 else row['Total_Coverage'] / row[year], axis=1)


    # print(f"Ratio DF:\n{calculate_ratios_DF[['antigen', 'Ratio']]}")

    return calculate_ratios_DF, total_coverage_df

# Example usage:
# result_df = calculate_coverage_and_ratios(inventory_DF_MCV, demand_80_MCV, V_a, year)


##### Initialize antigens/vaccines/prodicers

In [23]:
A = ["Measles", "Mumps", "Rubella"]
V = ["M", "MR", "MMR"]

A_v = {
    "M": ["Measles"],
    "MR": ["Measles", "Rubella"],
    "MMR": ["Measles", "Mumps", "Rubella"]
}

P = ["Biological_E", 
    "GSK","PT_Bio", 
    "Serum_Institute"
]

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"]
}

P_v = {
    "M": ["Serum_Institute", "PT_Bio"],
    "MR": ["Serum_Institute", "Biological_E"],
    "MMR": ["Serum_Institute", "GSK"]
}


#translate vaccine - antigen, to antigen - vaccine
V_a = {a: [v for v in A_v if a in A_v[v]] for a in A}

V_p = {p: [v for v in P_v if p in P_v[v]] for p in P}

P_a = {a: list(set(p for v in V_a[a] for p in P_v[v])) for a in A}

A_p = {p: [a for a in P_a if p in P_a[a]] for p in P}



In [24]:
P_a

{'Measles': ['GSK', 'Biological_E', 'Serum_Institute', 'PT_Bio'],
 'Mumps': ['GSK', 'Serum_Institute'],
 'Rubella': ['GSK', 'Serum_Institute', 'Biological_E']}

## TESTING - Measles Containing Vaccines Only

In [25]:
# Selecting only the rows for 'Measles', 'Mumps', and 'Rubella' in both datasets
demand_80_MCV = demand_80_final[demand_80_final['antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
tender_schedules_MCV = tender_schedules[tender_schedules['Antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
# i_start_MCV = i_start[i_start['Vaccine'].isin(['M', 'MR', 'MMR'])]
missed_doses_MCV = missed_doses[missed_doses['antigen'].isin(['Measles', 'Mumps', 'Rubella'])]
ratio_DF_MCV = pd.DataFrame()
inventory_DF_MCV = inventory_DF[inventory_DF['Vaccine'].isin(['M', 'MR', 'MMR'])]


##### Logic to translate vaccine totals to antigen coverage for later math

In [13]:
#logic to setup least covered antigens:
# Flatten the list of all antigens from all vaccines
all_antigens = [antigen for antigens in A_v.values() for antigen in antigens]
# Count the occurrences of each antigen
antigen_counts = Counter(all_antigens)
# least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)

In [14]:
for year in range(1, 4 + 1):  # Iterate through each year - short range for testing  range(1,len(demand_80_MCV.columns)-1)
    print("********************HAPPY NEW YEAR****************************")
    print("******************RETICULATING SPLINE**************************")
    print(f"Year: {year}")

    least_covered_antigens = sorted(antigen_counts.keys(), key=lambda x: antigen_counts[x], reverse=False)
    remaining_inventory = {}
    uncovered_demand = {}
    while least_covered_antigens:  # Iterate through each antigen, find what vaccines cover each antigen, least to greatest, update supply and demand
        print('###############################################################')
        antigen = least_covered_antigens.pop(0)
        print(f"serving antigen {antigen}")
        for vaccine, antigens in A_v.items():  # Iterate through A_v to check which vaccines cover the antigen
            if antigen in antigens and demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == antigen].iloc[0, year] >0:

                vaccine_inventory_value = inventory_DF_MCV.loc[inventory_DF_MCV.iloc[:, 0] == vaccine].iloc[0, 1]
                # print("iiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii")
                # print(f"Inventory for {vaccine} for year {year}: ", vaccine_inventory_value)

                antigen_demand_value = demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == antigen].iloc[0, year]
                # print(f"Demand for {antigen} for year {year}: ", antigen_demand_value)

                difference = vaccine_inventory_value - antigen_demand_value
                if difference >= 0: #
                    remaining_inventory[vaccine] = difference
                    decrement = antigen_demand_value
                else: 
                    remaining_inventory[vaccine] = 0
                    decrement = vaccine_inventory_value
                    uncovered_demand[antigen] = abs(difference)
                    print("------------------------------------------")
                    # print(f"Vaccine:3 {vaccine}, antigen: {antigen}")
                    print(f"uncovered demand for {antigen}: {[antigen]}")
                    #transfer uncovered demand to next year

                # print("^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^")
                inventory_DF_MCV.loc[inventory_DF_MCV.iloc[:, 0] == vaccine, inventory_DF_MCV.columns[1]] = remaining_inventory[vaccine]

                print("Decrementing antigen demands")
                for ant in antigens:
                    if demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == ant, demand_80_MCV.columns[year]].item() > 0:
                        # print(f"from {ant} demand, reducing demand for year {year} for antigen {ant} by {decrement}")
                        demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == ant, demand_80_MCV.columns[year]] = demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == ant].iloc[0, year] - decrement

        print(f"{len(antigen_counts)} antigens entered, only {least_covered_antigens} remain!")

    #update any uncovered demand, to next year. add uncovered demand to dosses_missed dict
    if 'uncovered_demand' in locals(): # Check if the variable exists
        while uncovered_demand:
            top = uncovered_demand.popitem()
            top_antigen = top[0]
            doses_missed = top[1]
            print(f"{doses_missed} doses missed for {top_antigen}")
            demand_80_MCV.loc[demand_80_MCV.iloc[:, 0] == top_antigen, demand_80_MCV.columns[year+1]] += doses_missed
            missed_doses_MCV.loc[missed_doses_MCV.iloc[:, 0] == top_antigen, missed_doses_MCV.columns[1]] += doses_missed
    else:
        print("no uncovered demand this year")
    
    #check ratio for supply/demand.
    #check at end of year for math reasons. if ratio is less than 1, schedule tender, perform search for vaccines, add inventory
    print(f"Checking ratio of supply to demand for antigens for year {year + 1}!")
    print()
    #pulls the current ratio of supply and demand. returns ratio_DF and antigen coverage DF
    ratio_DF_MCV, coverage_df = calculate_coverage_and_ratios(inventory_DF_MCV, demand_80_MCV, V_a, year+1)

    for index, row in ratio_DF_MCV.iterrows():
        if row['Ratio'] < 1:
            print(f"Ratio: {round(row.loc['Ratio'],2)}")
            print(f"Generating Tender for {row['antigen']}")
            #append F schedule for curreny year +1 to current year +1 + tender_length
            new_row = {'Antigen': row.loc['antigen'], 'Starting': year + 1, 'Ending': year + 4}
            tender_schedules_MCV = pd.concat([tender_schedules_MCV, pd.DataFrame([new_row])], ignore_index=True)
            #inventory search
        else:
            print(f"Ratio: {round(row.loc['Ratio'],2)}")
            print(f"Supply > demand for {row['antigen']}")



********************HAPPY NEW YEAR****************************
******************RETICULATING SPLINE**************************
Year: 1
###############################################################
serving antigen Mumps
Decrementing antigen demands
3 antigens entered, only ['Rubella', 'Measles'] remain!
###############################################################
serving antigen Rubella
Decrementing antigen demands
3 antigens entered, only ['Measles'] remain!
###############################################################
serving antigen Measles
Decrementing antigen demands
3 antigens entered, only [] remain!
Checking ratio of supply to demand for antigens for year 2!

Ratio: 2.1
Supply > demand for Measles
Ratio: 1.97
Supply > demand for Mumps
Ratio: 2.2
Supply > demand for Rubella
********************HAPPY NEW YEAR****************************
******************RETICULATING SPLINE**************************
Year: 2
###############################################################
ser